# RAG & Agent Evaluation — LangGraph Walkthrough

This notebook implements the evaluation concepts as **runnable LangGraph pipelines**, using mock tools/judges so everything runs with no API key.

**Structure:**
1. Part 1 — RAG retrieval metrics (Precision@K, Recall@K, MRR, nDCG)
2. Part 2 — RAG eval pipeline as a LangGraph graph (context relevance + faithfulness/groundedness)
3. Part 3 — Agent eval pipeline as a LangGraph graph (tool selection, task completion, trajectory scoring, unnecessary tool calls)

Each section starts with the interview question it answers, then the code that demonstrates it.

## Setup

```
pip install langgraph langchain-core
```

In [1]:
from typing import TypedDict, List, Dict, Optional
import math
from langgraph.graph import StateGraph, END

print("Imports OK")

Imports OK


---
# Part 1 — Retrieval Metrics

**Interview questions covered:**
- *How would you evaluate retrieval quality?*
- *What do Precision@K and Recall@K tell you?*
- *When would you use MRR or nDCG?*

**Approach:** build a labeled eval set (query → ground-truth relevant chunk IDs), run retrieval, score with rank-aware metrics — kept separate from generation quality so you can isolate *where* a RAG failure comes from.

In [2]:
def precision_at_k(retrieved: List[str], relevant: List[str], k: int) -> float:
    """Of the K chunks retrieved, how many are actually relevant?"""
    top_k = retrieved[:k]
    hits = [doc for doc in top_k if doc in relevant]
    return len(hits) / k if k > 0 else 0.0


def recall_at_k(retrieved: List[str], relevant: List[str], k: int) -> float:
    """Of all relevant chunks that exist, how many did we find in top K?"""
    top_k = retrieved[:k]
    hits = [doc for doc in top_k if doc in relevant]
    return len(hits) / len(relevant) if relevant else 0.0


def mrr(retrieved: List[str], relevant: List[str]) -> float:
    """Mean Reciprocal Rank: 1/rank of the FIRST relevant hit.
    Use when there's typically one correct/best chunk and you care how early it appears."""
    for rank, doc in enumerate(retrieved, start=1):
        if doc in relevant:
            return 1.0 / rank
    return 0.0


def dcg_at_k(retrieved: List[str], grades: Dict[str, int], k: int) -> float:
    score = 0.0
    for i, doc in enumerate(retrieved[:k]):
        rel = grades.get(doc, 0)
        score += rel / math.log2(i + 2)  # rank 1 -> log2(2)=1, avoids log2(1)=0 division
    return score


def ndcg_at_k(retrieved: List[str], grades: Dict[str, int], k: int) -> float:
    """Normalized DCG: use when relevance is GRADED (not binary) and multiple
    relevant chunks can coexist. Rewards relevant results appearing higher."""
    dcg = dcg_at_k(retrieved, grades, k)
    ideal_order = sorted(grades.values(), reverse=True)[:k]
    idcg = sum(rel / math.log2(i + 2) for i, rel in enumerate(ideal_order))
    return dcg / idcg if idcg > 0 else 0.0

print("Metric functions defined")

Metric functions defined


### Toy eval example

Query: *"What is the time complexity of BST delete?"*
- `retrieved_ids` — what the retriever actually returned, in rank order
- `relevant_ids` — ground truth (labeled once, reused across eval runs)
- `relevance_grades` — graded relevance (0-3) for nDCG; binary membership in `relevant_ids` is enough for Precision/Recall/MRR

In [3]:
query = "What is the time complexity of BST delete?"

retrieved_ids = ["chunk_3", "chunk_1", "chunk_7", "chunk_2", "chunk_9"]
relevant_ids  = ["chunk_1", "chunk_2", "chunk_5"]     # chunk_5 exists but wasn't retrieved -> hurts recall
relevance_grades = {
    "chunk_1": 3, "chunk_2": 2, "chunk_3": 0,
    "chunk_5": 3, "chunk_7": 1, "chunk_9": 0,
}

k = 5
print(f"Precision@{k}: {precision_at_k(retrieved_ids, relevant_ids, k):.2f}")
print(f"Recall@{k}:    {recall_at_k(retrieved_ids, relevant_ids, k):.2f}")
print(f"MRR:           {mrr(retrieved_ids, relevant_ids):.2f}")
print(f"nDCG@{k}:       {ndcg_at_k(retrieved_ids, relevance_grades, k):.2f}")

Precision@5: 0.40
Recall@5:    0.67
MRR:           0.50
nDCG@5:       0.51


**Reading the output:**
- Precision@5 = 0.40 → 2 of the 5 retrieved chunks were relevant (`chunk_1`, `chunk_2`). 3 were noise.
- Recall@5 = 0.67 → we found 2 of the 3 relevant chunks that exist; `chunk_5` was missed entirely.
- MRR = 0.50 → the first relevant chunk (`chunk_1`) appeared at rank 2, so 1/2.
- nDCG@5 factors in that `chunk_1` (grade 3) ranking below `chunk_3` (grade 0) is a bigger penalty than a small rank swap between two similarly-graded chunks would be.

---
# Part 2 — RAG Eval Pipeline as a LangGraph Graph

**Interview questions covered:**
- *How would you measure context relevance?*
- *How would you know the answer is grounded in the retrieved context?*

**Approach:** model the pipeline as a graph — `retrieve → judge_context_relevance → generate → check_faithfulness`. Each node writes eval signals into shared state. In production the "judge" functions below would be LLM calls (LLM-as-judge); here they're deterministic mocks so the notebook runs standalone.

- **Context relevance** = are the retrieved chunks actually about the query? (checked *before* generation)
- **Faithfulness/groundedness** = does the generated answer only assert things the context supports? (checked *after* generation, catches hallucination)

In [4]:
class RAGEvalState(TypedDict):
    query: str
    retrieved_chunks: List[Dict]
    relevance_judgments: Dict[str, str]     # chunk_id -> "relevant" / "irrelevant"
    answer: str
    claims: List[str]
    faithfulness_scores: Dict[str, bool]    # claim -> is it supported by context?
    context_relevance_score: float
    faithfulness_score: float

In [5]:
# ---- Mock "LLM-as-judge" functions ----
# In production: replace these with real LLM calls (e.g. "Is this chunk relevant to
# the query? yes/no", or "Is this claim entailed by the context? yes/no").

STOPWORDS = {"what", "is", "the", "of", "a", "an", "to", "in", "for", "and", "how"}

def mock_judge_relevance(query: str, chunk_text: str) -> str:
    query_terms = {w for w in query.lower().split() if w not in STOPWORDS and len(w) > 2}
    overlap = sum(1 for term in query_terms if term in chunk_text.lower())
    return "relevant" if overlap >= 1 else "irrelevant"

def mock_generate_answer(query: str, chunks: List[Dict]) -> str:
    relevant_text = " ".join(c["text"] for c in chunks)
    return f"Based on context: {relevant_text[:60]}..."

def mock_extract_claims(answer: str) -> List[str]:
    return [s.strip() for s in answer.split(".") if s.strip()]

def mock_check_claim_supported(claim: str, chunks: List[Dict]) -> bool:
    all_text = " ".join(c["text"] for c in chunks).lower()
    key_terms = [w for w in claim.lower().split() if len(w) > 4]
    return any(term in all_text for term in key_terms)

print("Mock judges defined")

Mock judges defined


In [6]:
def retrieve_node(state: RAGEvalState) -> RAGEvalState:
    # In a real pipeline this is a vector search call. Fixture data here for a
    # reproducible demo -- notice chunk_3 is deliberately off-topic.
    state["retrieved_chunks"] = [
        {"id": "c1", "text": "BST delete runs in O(h) time where h is tree height."},
        {"id": "c2", "text": "The successor node is the leftmost node of the right subtree."},
        {"id": "c3", "text": "Unrelated: Python lists are dynamic arrays."},
    ]
    return state


def judge_context_relevance_node(state: RAGEvalState) -> RAGEvalState:
    judgments = {c["id"]: mock_judge_relevance(state["query"], c["text"])
                 for c in state["retrieved_chunks"]}
    state["relevance_judgments"] = judgments
    relevant_count = sum(1 for v in judgments.values() if v == "relevant")
    state["context_relevance_score"] = relevant_count / len(judgments)
    return state


def generate_node(state: RAGEvalState) -> RAGEvalState:
    # Only feed relevant chunks into generation -- a common production pattern
    # (filter before you generate, don't just filter the eval after the fact).
    relevant_chunks = [c for c in state["retrieved_chunks"]
                        if state["relevance_judgments"][c["id"]] == "relevant"]
    state["answer"] = mock_generate_answer(state["query"], relevant_chunks)
    return state


def check_faithfulness_node(state: RAGEvalState) -> RAGEvalState:
    claims = mock_extract_claims(state["answer"])
    state["claims"] = claims
    scores = {c: mock_check_claim_supported(c, state["retrieved_chunks"]) for c in claims}
    state["faithfulness_scores"] = scores
    supported = sum(1 for v in scores.values() if v)
    state["faithfulness_score"] = supported / len(scores) if scores else 0.0
    return state

print("Nodes defined")

Nodes defined


In [7]:
builder = StateGraph(RAGEvalState)
builder.add_node("retrieve", retrieve_node)
builder.add_node("judge_context_relevance", judge_context_relevance_node)
builder.add_node("generate", generate_node)
builder.add_node("check_faithfulness", check_faithfulness_node)

builder.set_entry_point("retrieve")
builder.add_edge("retrieve", "judge_context_relevance")
builder.add_edge("judge_context_relevance", "generate")
builder.add_edge("generate", "check_faithfulness")
builder.add_edge("check_faithfulness", END)

rag_eval_graph = builder.compile()
print("Graph compiled: retrieve -> judge_context_relevance -> generate -> check_faithfulness -> END")

Graph compiled: retrieve -> judge_context_relevance -> generate -> check_faithfulness -> END


In [8]:
result = rag_eval_graph.invoke({"query": "What is the time complexity of BST delete?"})

print("Context relevance judgments:", result["relevance_judgments"])
print(f"Context relevance score:      {result['context_relevance_score']:.2f}")
print()
print("Answer:", result["answer"])
print("Faithfulness scores:", result["faithfulness_scores"])
print(f"Faithfulness score:           {result['faithfulness_score']:.2f}")

Context relevance judgments: {'c1': 'relevant', 'c2': 'irrelevant', 'c3': 'irrelevant'}
Context relevance score:      0.33

Answer: Based on context: BST delete runs in O(h) time where h is tree height....
Faithfulness scores: {'Based on context: BST delete runs in O(h) time where h is tree height': True}
Faithfulness score:           1.00


**Reading the output:**
- `c3` (Python lists — off-topic) is correctly flagged `irrelevant` → this is the **context relevance** signal, computed *before* generation even happens.
- The **faithfulness score** checks the *opposite direction*: given what got generated, is every claim traceable back to the retrieved context? A score of 1.00 here means no hallucination was detected — everything the model said maps to a retrieved chunk.
- These two scores catch **different failure modes**: bad context relevance = retriever problem. Bad faithfulness despite good context = generator problem (model ignored context / made things up).

---
# Part 3 — Agent Eval Pipeline as a LangGraph Graph

**Interview questions covered:**
- *How would you evaluate whether the agent selected the right tool?*
- *How would you measure task completion?*
- *What happens when the agent takes the wrong action?*
- *How would you evaluate multi-step tasks?*
- *How would you detect unnecessary tool calls?*

**Approach:** run an agent tool-calling loop (`plan_and_act`, looping via a conditional edge) that builds a **trace**, then hand the trace to a separate `evaluate` node. Keeping the agent loop and the eval logic as separate nodes mirrors real practice — the agent doesn't grade itself; a separate harness does, against a **gold trajectory** defined ahead of time.

In [9]:
class AgentEvalState(TypedDict):
    query: str
    gold_tool_sequence: List[str]     # ground truth trajectory, defined by the eval harness
    trace: List[Dict]                  # [{"tool", "args", "result", "used"}]
    current_step: int
    final_answer: Optional[str]
    task_success: Optional[bool]
    tool_selection_correct: List[bool]
    unnecessary_calls: List[str]

In [10]:
# ---- Mock tools the agent can call ----
def tool_search_docs(args):
    return f"docs about {args.get('topic', '?')}"

def tool_calculator(args):
    return str(eval(args.get("expr", "0")))  # toy only -- never eval() untrusted input in real code

def tool_send_email(args):
    return f"email sent to {args.get('to', '?')}"

TOOLS = {
    "search_docs": tool_search_docs,
    "calculator": tool_calculator,
    "send_email": tool_send_email,
}
print("Tools registered:", list(TOOLS.keys()))

Tools registered: ['search_docs', 'calculator', 'send_email']


In [11]:
# ---- Mock planner ----
# In production this is an LLM call deciding the next action given state + tool results.
# This plan deliberately repeats a call, to demonstrate "unnecessary tool call" detection.
def mock_plan_next_step(state: AgentEvalState) -> Optional[Dict]:
    plan = [
        {"tool": "search_docs", "args": {"topic": "BST delete complexity"}},
        {"tool": "search_docs", "args": {"topic": "BST delete complexity"}},  # redundant repeat
        {"tool": "calculator", "args": {"expr": "2**10"}},
    ]
    step = state["current_step"]
    return plan[step] if step < len(plan) else None

In [12]:
def plan_and_act_node(state: AgentEvalState) -> AgentEvalState:
    next_call = mock_plan_next_step(state)
    if next_call is None:
        state["final_answer"] = "Task complete."
        return state

    tool_name, args = next_call["tool"], next_call["args"]
    result = TOOLS[tool_name](args)

    # A call is "unnecessary" if the exact same (tool, args) already happened --
    # the result would be identical, so it added no new information.
    is_duplicate = any(t["tool"] == tool_name and t["args"] == args for t in state["trace"])

    state["trace"].append({
        "tool": tool_name, "args": args, "result": result, "used": not is_duplicate,
    })
    state["current_step"] += 1
    return state


def should_continue(state: AgentEvalState) -> str:
    # This is the loop: keep acting until the planner signals completion.
    return "plan_and_act" if state.get("final_answer") is None else "evaluate"


def evaluate_node(state: AgentEvalState) -> AgentEvalState:
    actual_tools = [t["tool"] for t in state["trace"]]
    gold_tools = state["gold_tool_sequence"]

    # Step-wise trajectory match against the gold sequence -- this is how you
    # evaluate MULTI-STEP tasks: score each step, not just the final answer.
    correctness = [
        (actual_tools[i] if i < len(actual_tools) else None) == gold_tools[i]
        for i in range(len(gold_tools))
    ]
    state["tool_selection_correct"] = correctness

    # Flags raised during the trace itself -- redundant calls with no new info.
    state["unnecessary_calls"] = [t["tool"] for t in state["trace"] if not t["used"]]

    # Task completion = reached a final answer AND every gold step was matched.
    # (In a real harness you'd also verify END STATE, e.g. did the email actually send.)
    state["task_success"] = (
        state["final_answer"] is not None and sum(correctness) == len(gold_tools)
    )
    return state

print("Nodes defined")

Nodes defined


In [13]:
builder = StateGraph(AgentEvalState)
builder.add_node("plan_and_act", plan_and_act_node)
builder.add_node("evaluate", evaluate_node)

builder.set_entry_point("plan_and_act")
builder.add_conditional_edges(
    "plan_and_act",
    should_continue,
    {"plan_and_act": "plan_and_act", "evaluate": "evaluate"},
)
builder.add_edge("evaluate", END)

agent_eval_graph = builder.compile()
print("Graph compiled: plan_and_act (loops) -> evaluate -> END")

Graph compiled: plan_and_act (loops) -> evaluate -> END


In [14]:
initial_state: AgentEvalState = {
    "query": "Look up BST delete complexity and compute 2^10",
    "gold_tool_sequence": ["search_docs", "calculator"],   # the IDEAL 2-step trajectory
    "trace": [],
    "current_step": 0,
    "final_answer": None,
    "task_success": None,
    "tool_selection_correct": [],
    "unnecessary_calls": [],
}

result = agent_eval_graph.invoke(initial_state)

print("=== Trace ===")
for i, step in enumerate(result["trace"]):
    print(f"  Step {i}: {step['tool']}({step['args']}) -> {step['result']!r}  | used={step['used']}")

print()
print("=== Eval ===")
print("Tool selection correctness vs gold:", result["tool_selection_correct"])
print("Unnecessary calls detected:        ", result["unnecessary_calls"])
print("Task success:                      ", result["task_success"])
print("Final answer:                      ", result["final_answer"])

=== Trace ===
  Step 0: search_docs({'topic': 'BST delete complexity'}) -> 'docs about BST delete complexity'  | used=True
  Step 1: search_docs({'topic': 'BST delete complexity'}) -> 'docs about BST delete complexity'  | used=False
  Step 2: calculator({'expr': '2**10'}) -> '1024'  | used=True

=== Eval ===
Tool selection correctness vs gold: [True, False]
Unnecessary calls detected:         ['search_docs']
Task success:                       False
Final answer:                       Task complete.


**Reading the output:**
- **Tool selection accuracy** — step 0 matches gold (`search_docs`), step 1 doesn't (gold expected `calculator`, agent repeated `search_docs`) → `[True, False]`. This is exactly the "right tool, wrong step" signal from the eval questions.
- **Unnecessary tool calls** — the duplicate `search_docs` call is flagged because its `(tool, args)` pair already appeared in the trace with no new information gained. In a real system you'd also track cost/latency here, since redundant calls are a common source of runaway agent cost.
- **Task completion** — `False`, because even though the agent reached a final answer, its trajectory deviated from the gold path at step 1. This shows why **task success ≠ "did it stop without error"** — you need the full trajectory check, not just presence of a final message.
- **What happens on a wrong action** — nothing in this toy agent *catches* the redundant call mid-loop; it just gets executed and flagged afterward by the eval harness. A more robust agent would validate before calling (e.g., check "have I already fetched this?") so the wrong action never happens, or self-correct by inspecting the trace so far. That's the gap between what this demo evaluates and what a production-grade agent would additionally prevent.

---
# Part 4 — Full Agent Metric Suite on a Realistic Scenario

**Scenario:** a customer-support agent with three tools — `search_orders`, `get_refund_policy`, `issue_refund`. User asks: *"My order #4521 arrived broken, can I get a refund?"*

This scenario is deliberately built to exercise every metric in one trace:
- A **typo'd order ID** on the first call → a real tool error
- The agent **retries with the correct ID** → a recovery signal
- A **redundant repeat** of the policy lookup → an unnecessary call
- A real **mutation to the mock "order database"** → something end-state verification can check independently of what the agent claims

**Metrics covered in this part:**
1. Tool Selection Accuracy
2. Tool Call Correctness (Arguments)
3. Unnecessary / Redundant Tool Calls
4. Step-wise Accuracy
5. Task Success Rate
6. Trajectory Match (fuzzy)
7. End-State Verification
8. Recovery / Self-Correction Rate
9. Cost / Efficiency proxy (call count as a stand-in for tokens/latency)

### State schema for this scenario

Defines the LangGraph state for the refund-agent run: `order_db` (the mutable mock database used later for end-state verification), `gold_trajectory` (the tool set the task is expected to exercise), and a `trace` that — unlike Parts 2-3 — also records `status` (`"ok"`/`"error"`) and `redundant` per step, since those two fields are exactly what the metrics further down key off of.

In [30]:
class RefundAgentState(TypedDict):
    query: str
    order_db: Dict[int, Dict]       # mock "system of record" -- what end-state verification checks
    gold_trajectory: List[str]       # gold tool set the task should exercise
    trace: List[Dict]                # [{"step","tool","args","result","status","redundant"}]
    current_step: int
    final_answer: Optional[str]
    max_steps: int

### Tools + mock system of record

`order_db` is a plain dict standing in for a real database. `issue_refund` **mutates** it — this is what lets us check End-State Verification independently of the agent's own final message.

In [31]:
def tool_search_orders(args, db):
    order_id = args.get("order_id")
    order = db.get(order_id)
    if order is None:
        return {"status": "error", "result": f"order {order_id} not found"}
    return {"status": "ok", "result": order}

def tool_get_refund_policy(args, db):
    return {"status": "ok", "result": "Orders damaged on arrival are eligible for full refund."}

def tool_issue_refund(args, db):
    order_id = args.get("order_id")
    amount = args.get("amount")
    order = db.get(order_id)
    if order is None:
        return {"status": "error", "result": f"cannot refund: order {order_id} not found"}
    order["refund_issued"] = True     # <-- the actual state mutation
    order["refund_amount"] = amount
    return {"status": "ok", "result": f"refund of ${amount} issued for order {order_id}"}

TOOLS = {
    "search_orders": tool_search_orders,
    "get_refund_policy": tool_get_refund_policy,
    "issue_refund": tool_issue_refund,
}
print("Tools registered:", list(TOOLS.keys()))

Tools registered: ['search_orders', 'get_refund_policy', 'issue_refund']


### Mock planner with realistic failure modes

Unlike Parts 2-3's simpler plans, this one deliberately includes an error (typo'd order ID) followed by a correction, plus a redundant call -- so every metric below has something real to measure.

In [32]:
def mock_plan_next_step(state: RefundAgentState) -> Optional[Dict]:
    plan = [
        {"tool": "search_orders", "args": {"order_id": 4251}},         # typo -> will error
        {"tool": "search_orders", "args": {"order_id": 4521}},         # corrected -> recovery
        {"tool": "get_refund_policy", "args": {}},
        {"tool": "get_refund_policy", "args": {}},                      # redundant repeat
        {"tool": "issue_refund", "args": {"order_id": 4521, "amount": 45}},
    ]
    step = state["current_step"]
    return plan[step] if step < len(plan) else None

### Agent loop + graph

`plan_and_act_node` executes the next planned tool call against `order_db`, marks it `redundant` if the identical `(tool, args)` pair already **succeeded** earlier in the trace, and appends the full record — including `status` — to `trace`. `should_continue` loops back to `plan_and_act` until the planner runs out of steps (or `max_steps` is hit), then ends. The graph itself is a single self-looping node, simpler than Part 3's two-node design, because evaluation is done entirely separately below rather than as a graph node.

In [33]:
def plan_and_act_node(state: RefundAgentState) -> RefundAgentState:
    next_call = mock_plan_next_step(state)
    if next_call is None or state["current_step"] >= state["max_steps"]:
        state["final_answer"] = "Your refund of $45 has been issued."
        return state

    tool_name, args = next_call["tool"], next_call["args"]
    outcome = TOOLS[tool_name](args, state["order_db"])

    # A call is redundant if the SAME (tool, args) already succeeded earlier in the trace
    is_duplicate = any(
        t["tool"] == tool_name and t["args"] == args and t["status"] == "ok"
        for t in state["trace"]
    )

    state["trace"].append({
        "step": state["current_step"],
        "tool": tool_name,
        "args": args,
        "result": outcome["result"],
        "status": outcome["status"],      # "ok" or "error"
        "redundant": is_duplicate,
    })
    state["current_step"] += 1
    return state


def should_continue(state: RefundAgentState) -> str:
    return "plan_and_act" if state.get("final_answer") is None else END


builder = StateGraph(RefundAgentState)
builder.add_node("plan_and_act", plan_and_act_node)
builder.set_entry_point("plan_and_act")
builder.add_conditional_edges("plan_and_act", should_continue, {"plan_and_act": "plan_and_act", END: END})
refund_agent_graph = builder.compile()
print("Graph compiled")

Graph compiled


### Run the scenario

Seeds `order_db` with one order (`4521`, `refund_issued: False`) and a `gold_trajectory` of the three tools the task should exercise, then invokes the graph. `trace` and `order_db_after` — the two artifacts every metric below is computed from — are pulled out of the result here.

In [35]:
state = {
    "query": "My order #4521 arrived broken, can I get a refund?",
    "order_db": {4521: {"item": "Headphones", "price": 45, "refund_issued": False}},
    "gold_trajectory": ["search_orders", "get_refund_policy", "issue_refund"],
    "trace": [],
    "current_step": 0,
    "final_answer": None,
    "max_steps": 6,
}

In [36]:
next_call = mock_plan_next_step(state)
next_call

{'tool': 'search_orders', 'args': {'order_id': 4251}}

In [19]:
initial_state: RefundAgentState = {
    "query": "My order #4521 arrived broken, can I get a refund?",
    "order_db": {4521: {"item": "Headphones", "price": 45, "refund_issued": False}},
    "gold_trajectory": ["search_orders", "get_refund_policy", "issue_refund"],
    "trace": [],
    "current_step": 0,
    "final_answer": None,
    "max_steps": 6,
}

result = refund_agent_graph.invoke(initial_state)
trace = result["trace"]
order_db_after = result["order_db"]

print("=== TRACE ===")
for t in trace:
    print(f"  step {t['step']}: {t['tool']}({t['args']}) -> {t['result']!r}  [{t['status']}]  redundant={t['redundant']}")

print("\nFinal answer:", result["final_answer"])

=== TRACE ===
  step 0: search_orders({'order_id': 4251}) -> 'order 4251 not found'  [error]  redundant=False
  step 1: search_orders({'order_id': 4521}) -> {'item': 'Headphones', 'price': 45, 'refund_issued': True, 'refund_amount': 45}  [ok]  redundant=False
  step 2: get_refund_policy({}) -> 'Orders damaged on arrival are eligible for full refund.'  [ok]  redundant=False
  step 3: get_refund_policy({}) -> 'Orders damaged on arrival are eligible for full refund.'  [ok]  redundant=True
  step 4: issue_refund({'order_id': 4521, 'amount': 45}) -> 'refund of $45 issued for order 4521'  [ok]  redundant=False

Final answer: Your refund of $45 has been issued.


**Reading the trace:** step 0 fails (wrong order ID), step 1 corrects it and succeeds, steps 2-3 look up policy twice (only the first was necessary), step 4 issues the refund. This one trace contains an error, a recovery, and a redundancy — exactly the raw material each metric below needs.

### Computing all 9 metrics from the trace

Each metric reads from `trace` and `order_db_after` — nothing here needs a new agent run, which mirrors how a real eval harness works: run once, score many ways.

### 1. Tool Selection Accuracy

Checks *coverage*, not order: does the set of tools the agent actually called include every tool in the gold set? This is the loosest of the nine metrics — it would pass even if the agent called tools in the wrong order or threw in extra redundant calls, which is exactly why the metrics that follow narrow in on those specific failure modes.

In [20]:
gold_tools_set = set(result["gold_trajectory"])
called_tools_set = {t["tool"] for t in trace}

# 1. Tool Selection Accuracy -- did the agent's trace cover every tool the gold set required?
tool_selection_coverage = len(gold_tools_set & called_tools_set) / len(gold_tools_set)
print(f"1. Tool Selection Accuracy: {tool_selection_coverage:.2f}   (gold tools = {gold_tools_set})")

1. Tool Selection Accuracy: 1.00   (gold tools = {'search_orders', 'issue_refund', 'get_refund_policy'})


### 2. Tool Call Correctness (Arguments)

Passing metric 1 only confirms the *right tools* were called — this checks whether the *arguments* passed to `issue_refund` were actually valid (correct `order_id`, a positive `amount`). A tool can be correctly selected and still be called with wrong or malformed arguments; this metric catches that separately, on the calls that actually succeeded.

In [21]:
# 2. Tool Call Correctness (Arguments) -- right tool, but were the ARGS actually right?
issue_calls = [t for t in trace if t["tool"] == "issue_refund" and t["status"] == "ok"]
correct_args = all(
    c["args"].get("order_id") == 4521 and c["args"].get("amount", 0) > 0
    for c in issue_calls
)
print(f"2. Tool Call Correctness (issue_refund args valid): {correct_args}")

2. Tool Call Correctness (issue_refund args valid): True


### 3. Unnecessary / Redundant Tool Calls

Simply counts the trace entries `plan_and_act_node` already flagged `redundant` — a call whose `(tool, args)` pair had already **succeeded** earlier in the same trace, adding no new information. Here that's the duplicate `get_refund_policy` lookup.

In [22]:
# 3. Unnecessary / Redundant Tool Calls -- same (tool, args) succeeding more than once
redundant_calls = [t for t in trace if t["redundant"]]
print(f"3. Unnecessary Tool Calls: {len(redundant_calls)}  -> {[t['tool'] for t in redundant_calls]}")

3. Unnecessary Tool Calls: 1  -> ['get_refund_policy']


### 4. Step-wise Accuracy

Fraction of *all* trace steps that were both `status == "ok"` and not `redundant` — i.e. steps that made real forward progress. Unlike Tool Selection Accuracy (which only checks coverage of the tool set), this penalizes wasted steps — errors and redundant calls — at the individual-step level.

In [23]:
# 4. Step-wise Accuracy -- fraction of steps that were both successful AND non-redundant
good_steps = [t for t in trace if t["status"] == "ok" and not t["redundant"]]
step_accuracy = len(good_steps) / len(trace)
print(f"4. Step-wise Accuracy: {step_accuracy:.2f}   ({len(good_steps)}/{len(trace)} steps)")

4. Step-wise Accuracy: 0.60   (3/5 steps)


### 5. Task Success Rate

For this single run, checks the ground-truth outcome directly: was `refund_issued` actually set, and does `refund_amount` match the expected `45`? In a real eval suite this boolean would be computed per task and averaged across many tasks to get an actual *rate* — here it's just the one-task building block.

In [24]:
# 5. Task Success Rate -- for THIS task, did the intended outcome actually happen?
# (In a real eval you'd average this boolean across many tasks to get a rate.)
task_success = order_db_after[4521]["refund_issued"] and order_db_after[4521]["refund_amount"] == 45
print(f"5. Task Success (this task): {task_success}")

5. Task Success (this task): True


### 6. Trajectory Match (fuzzy / set-based)

Compares the *set* of tools that ended in `"ok"` against the gold tool set, ignoring order, retries, and errors entirely. Deliberately looser than an exact-sequence match, which would fail here purely because of the typo'd first call and the redundant lookup — even though the agent still did the right things overall.

In [25]:
# 6. Trajectory Match (fuzzy / set-based) -- ignores order and retries, checks the
# set of SUCCESSFUL tool types matches the gold set. A stricter exact-sequence
# match would fail here because of the extra error + redundant steps.
actual_ok_tools = {t["tool"] for t in trace if t["status"] == "ok"}
trajectory_fuzzy_match = actual_ok_tools == gold_tools_set
print(f"6. Trajectory Match (fuzzy, set-based): {trajectory_fuzzy_match}")

6. Trajectory Match (fuzzy, set-based): True


### 7. End-State Verification

The most important check in this suite: instead of trusting `result['final_answer']` — a string the agent generated, which could be wrong or fabricated — this reads `order_db_after`, the actual mutated "system of record," directly, to confirm the refund really was recorded and not just claimed.

In [26]:
# 7. End-State Verification -- check the actual "system of record" directly,
# NOT the agent's self-reported final_answer string. This is the check that would
# catch a case where the agent claims success but the mutation never happened.
end_state_verified = (
    order_db_after.get(4521, {}).get("refund_issued") is True
    and order_db_after.get(4521, {}).get("refund_amount") == 45
)
print(f"7. End-State Verification (checked DB directly): {end_state_verified}")
print(f"   Agent claimed: {result['final_answer']!r}")
print(f"   DB actually shows: {order_db_after[4521]}")

7. End-State Verification (checked DB directly): True
   Agent claimed: 'Your refund of $45 has been issued.'
   DB actually shows: {'item': 'Headphones', 'price': 45, 'refund_issued': True, 'refund_amount': 45}


### 8. Recovery / Self-Correction Rate

For every trace entry with `status == "error"`, checks whether a *later* entry called the *same tool* and succeeded — that's the signal the agent noticed its own mistake and corrected it, rather than giving up or blindly repeating the same bad call. Reported as `recovered / total errors`, or `None` when there were no errors to recover from.

In [27]:
# 8. Recovery / Self-Correction Rate -- for each error, was there a LATER successful
# call to the SAME tool? That's the signal the agent noticed and corrected itself.
errors = [t for t in trace if t["status"] == "error"]
recovered = sum(
    1 for err in errors
    if any(t["step"] > err["step"] and t["tool"] == err["tool"] and t["status"] == "ok" for t in trace)
)
recovery_rate = recovered / len(errors) if errors else None
print(f"8. Recovery Rate: {recovered}/{len(errors)} errors recovered  -> {recovery_rate}")

8. Recovery Rate: 1/1 errors recovered  -> 1.0


### 9. Cost / Efficiency proxy

Treats redundant calls and errors as "wasted" and computes `1 - wasted/total` as a lightweight stand-in for a real cost metric (token usage, latency) that in production you'd pull from tracing infrastructure instead — see the Databricks mapping table below for where that data would actually come from.

In [28]:
# 9. Cost / Efficiency proxy -- in production this would be tokens/latency (e.g. from
# MLflow Tracing span metadata); call count is a simple stand-in for the same idea.
total_calls = len(trace)
wasted_calls = len(redundant_calls) + len(errors)
efficiency = 1 - (wasted_calls / total_calls)
print(f"9. Cost/Efficiency proxy: {total_calls} calls total, {wasted_calls} wasted (error+redundant)  -> efficiency={efficiency:.2f}")

9. Cost/Efficiency proxy: 5 calls total, 2 wasted (error+redundant)  -> efficiency=0.60


**Reading the results together:**

- Tool selection and end-state verification both come back **positive** — the agent used every tool it needed to and the refund really was recorded in the "database," not just claimed.
- But step-wise accuracy is only **0.60** and cost efficiency only **0.60** — because 2 of the 5 steps were waste (1 error, 1 redundant call). This is exactly why **outcome metrics alone can hide inefficiency**: the task succeeded, but it took 40% more calls than necessary to get there.
- Recovery rate of **1.0** is a genuinely good sign — the one error that occurred was caught and corrected, not left to fail silently or retried blindly with the same bad input.
- This is the core argument for running **step-level and trajectory-level metrics together**: task success alone would have reported "pass" and told you nothing about the wasted policy lookup or the initial wrong order ID.

### Databricks implementation notes for this scenario

| Metric | How this maps onto Databricks |
|---|---|
| Tool Selection Accuracy | Gold tool set stored in a Delta table; compared against tool spans captured by MLflow Tracing via `mlflow.langchain.autolog()` |
| Tool Call Correctness | `search_orders` / `issue_refund` defined as **Unity Catalog Functions** with typed signatures (`order_id: INT`) — a malformed call fails validation before it ever executes |
| Unnecessary Tool Calls | Query the MLflow trace table for duplicate `(tool, inputs)` spans within the same `request_id` |
| Step-wise Accuracy | Custom per-step scorer passed into `mlflow.genai.evaluate()`, iterating trace spans |
| Task Success Rate | Eval set of `(ticket_id, expected_outcome)` in Delta; agent run via a Databricks Job; pass/fail logged as an MLflow metric, trended in a Lakeview dashboard |
| Trajectory Match | Gold trajectories as JSON in Delta; a Unity Catalog-registered comparison function invoked as a custom `mlflow.genai.evaluate()` metric |
| End-State Verification | Delta Lake time-travel diff (`VERSION AS OF`) comparing the `order_db`-equivalent table before and after the agent run, inside the same Databricks Workflow |
| Recovery Rate | MLflow Tracing marks error-status spans automatically; a scheduled notebook pattern-matches them against later successful spans in the same trace tree |
| Cost / Efficiency | MLflow Tracing captures token usage and latency **per span automatically** — no manual instrumentation needed, just query the trace table |

Databricks now recommends **MLflow 3's `mlflow.genai.evaluate()`** with built-in LLM judges (Mosaic AI Agent Evaluation) over the older `mlflow.evaluate()` API — the pattern above uses that newer surface.

---
## Summary — mapping back to the interview answers

| Question | Where in this notebook |
|---|---|
| How would you evaluate retrieval quality? | Part 1 — full metric suite run against a labeled query |
| Precision@K / Recall@K | Part 1 — `precision_at_k`, `recall_at_k` |
| When MRR vs nDCG? | Part 1 — `mrr` (single best answer) vs `ndcg_at_k` (graded, multiple relevant) |
| Context relevance | Part 2 — `judge_context_relevance_node`, scored *before* generation |
| Groundedness / faithfulness | Part 2 — `check_faithfulness_node`, claim-by-claim check *after* generation |
| Right tool selected? | Part 3 — `tool_selection_correct`; Part 4 — Tool Selection Accuracy on a richer trace |
| Task completion | Part 3 — `task_success`; Part 4 — Task Success Rate checked against real state mutation |
| Wrong action taken | Part 3 — discussion only; Part 4 — a real tool error + measured **Recovery Rate** |
| Multi-step evaluation | Part 3 — step-wise trajectory scoring; Part 4 — Step-wise Accuracy + fuzzy Trajectory Match |
| Unnecessary tool calls | Part 3 — `unnecessary_calls`; Part 4 — redundant call detection on a realistic trace |
| Tool call argument correctness | Part 4 — `issue_refund` args validated independently of tool selection |
| End-state verification | Part 4 — checks the mock "database" directly, not the agent's self-reported answer |
| Cost / latency proxy | Part 4 — wasted-call ratio as a stand-in for token/latency tracking |
| Databricks implementation | Part 4 — metric-by-metric mapping to MLflow Tracing, `mlflow.genai.evaluate()`, Unity Catalog Functions, and Delta time-travel |